# NB-10: VAE Decoder -- Latent to Video Reconstruction

## Learning Objectives

- Trace Decoder3d's 4-level spatial upsampling path from 8x8 to 64x64
- Understand the Resample channel-halving asymmetry and the decoder's `in_dim //= 2` compensation
- Verify encode-decode round-trip preserves video dimensions

## Prerequisites

- **Prior notebooks:** NB-08 (CausalConv3d, ResidualBlock, AttentionBlock), NB-09 (Encoder3d, reparameterization)
- **Assumed concepts:** Spatial upsampling, nearest-neighbor interpolation

## Concept Map

- **Decoder3d** -> reconstructs video from DiT-produced latents (Phase 6 Integration)
- **Round-trip** -> confirms encoder/decoder are dimensionally symmetric
- **Channel halving** -> key decoder-specific pattern not present in encoder

In [ ]:
import sys
import types
import importlib.util
import pathlib

_here = pathlib.Path().resolve()
PROJECT_ROOT = None
for _candidate in [_here] + list(_here.parents)[:6]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
sys.modules["diffsynth"] = types.ModuleType("diffsynth")
sys.modules["diffsynth.models"] = types.ModuleType("diffsynth.models")
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

_vae_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_vae.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_vae", _vae_path)
vae_mod = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_vae"] = vae_mod
_spec.loader.exec_module(vae_mod)

from diffsynth.models.wan_video_vae import (
    Encoder3d, Decoder3d, CausalConv3d, ResidualBlock, AttentionBlock, Resample, RMS_norm
)
import torch
import torch.nn as nn
import torch.nn.functional as F

## Decoder3d Forward -- 4-Level Spatial Upsample

```
Decoder3d Forward -- 4-Level Spatial Upsample
==============================================

LATENT [B, z_dim=4, T, H/8=8, W/8=8]  (input: 1, 4, 5, 8, 8)
    |
    v
conv1: CausalConv3d(z_dim=4, 4*dim=128, k=3, p=1)
    |                                        [1, 128, 5, 8, 8]
    v
MIDDLE: ResBlock(128) -> AttentionBlock(128) -> ResBlock(128)
    |                                        [1, 128, 5, 8, 8]
    v
+-- Level 0: ResBlock(128->128) x4 + Resample(128, upsample3d)
|            Resample outputs 128//2=64 channels!
|                                            [1, 64, 5, 16, 16]  (spatial x2)
+-- Level 1: in_dim=128//2=64
|            ResBlock(64->128), then ResBlock(128->128) x3
|            + Resample(128, upsample3d)
|            Resample outputs 128//2=64 channels!
|                                            [1, 64, 5, 32, 32]  (spatial x2)
+-- Level 2: in_dim=128//2=64
|            ResBlock(64->64) x4 + Resample(64, upsample2d)
|            Resample outputs 64//2=32 channels!
|                                            [1, 32, 5, 64, 64]  (spatial x2)
+-- Level 3: in_dim=64//2=32
|            ResBlock(32->32) x4 [NO upsample]
    |                                        [1, 32, 5, 64, 64]
    v
HEAD: RMS_norm -> SiLU -> CausalConv3d(32, 3, k=3, p=1)
    |                                        [1, 3, 5, 64, 64]
    v
VIDEO [B, 3, T, H, W]  (reconstructed)
```

> **Key insight:** Each Resample (upsample) internally halves channels via `Conv2d(dim, dim//2)`,
> so the decoder compensates at levels 1-3 with `in_dim = in_dim // 2` (lines 770-771).
> The inner loop runs `num_res_blocks + 1 = 4` ResBlocks per level (line 772).

## 1. Channel Arithmetic: The Resample Halving Pattern

This is the key architectural insight of the decoder (Pitfall 3 from research).

**The asymmetry:**
- **Encoder Resample (downsample2d/3d):** `Conv2d(dim, dim, 3, stride=2)` -- channels **preserved**
- **Decoder Resample (upsample2d/3d):** `Upsample(2x) + Conv2d(dim, dim//2, 3, padding=1)` -- channels **halved**

This means the actual channel count entering the next level is half of what the `dims` list would suggest.

Source: `wan_video_vae.py`, lines 770-771:
```python
if i == 1 or i == 2 or i == 3:
    in_dim = in_dim // 2
```

**Why:** After `Resample(128, upsample3d)` the output has 64 channels, not 128.
The next level's ResidualBlocks must start with `in_dim=64`, not 128.

### Channel Trace Table

With `dim=32, dim_mult=[1,2,4,4]`: `dims = [128, 128, 128, 64, 32]`

| Level | dims[i]->dims[i+1] | in_dim (halved?) | First ResBlock | After Resample |
|-------|--------------------|------------------|----------------|----------------|
| 0     | 128->128           | 128 (no halving) | ResBlock(128->128) x4 | Resample(128) -> 64 |
| 1     | 128->128           | 128//2=64        | ResBlock(64->128), then (128->128) x3 | Resample(128) -> 64 |
| 2     | 128->64            | 128//2=64        | ResBlock(64->64) x4  | Resample(64) -> 32  |
| 3     | 64->32             | 64//2=32         | ResBlock(32->32) x4  | (no Resample)       |

Note: Level 1's first ResBlock has `in_dim=64 != out_dim=128`, so it uses the 1x1 shortcut projection.
The `num_res_blocks + 1 = 4` count comes from source line 772: `for _ in range(num_res_blocks + 1)`.

In [ ]:
# Source: wan_video_vae.py, lines 736-787
# VAE reduced config (must match encoder)
dim = 32
z_dim = 4
dim_mult = [1, 2, 4, 4]
num_res_blocks = 3  # NOTE: inner loop runs num_res_blocks+1 = 4 ResBlocks per level (line 772)

dec = Decoder3d(dim=dim, z_dim=z_dim, dim_mult=dim_mult,
                num_res_blocks=num_res_blocks,
                temperal_upsample=[True, True, False])

total_params = sum(p.numel() for p in dec.parameters())
print(f"Decoder3d parameters: {total_params:,}")
print(f"\nConfig: dim={dim}, z_dim={z_dim}, dim_mult={dim_mult}")

# Verify dims computation
dims = [dim * u for u in [dim_mult[-1]] + dim_mult[::-1]]
print(f"dims = [dim * u for u in [dim_mult[-1]] + dim_mult[::-1]]")
print(f"      = {dims}")
print(f"Channel progression: {dims[0]} -> {dims[-1]} (highest to lowest)")
print(f"ResBlocks per level: {num_res_blocks + 1} (range(num_res_blocks + 1))")

print(f"\nSub-module breakdown:")
print(f"  conv1: {sum(p.numel() for p in dec.conv1.parameters()):,} params")
print(f"  middle ({len(dec.middle)} layers): {sum(p.numel() for p in dec.middle.parameters()):,} params")
print(f"  upsamples ({len(list(dec.upsamples))} layers): {sum(p.numel() for p in dec.upsamples.parameters()):,} params")
print(f"  head: {sum(p.numel() for p in dec.head.parameters()):,} params")

# Verify layer count: 4 levels x (4 ResBlocks) + 3 Resamples = 19 layers
assert len(list(dec.upsamples)) == 19, f"Expected 19 upsample layers, got {len(list(dec.upsamples))}"

## 2. Step-by-Step Forward Pass Trace

We trace the decoder forward pass level by level, paying close attention to channel
counts after each Resample. The key: **Resample(upsample) halves channels internally.**

Each level contains 4 ResBlocks + 1 Resample (except Level 3 which has no Resample),
for a total of 5 layers per level (4 for the last).

In [ ]:
# Source: wan_video_vae.py, lines 789-838
# Step-by-step Decoder3d forward pass trace

dec.eval()
z = torch.randn(1, z_dim, 5, 8, 8)
x = z.clone()
print(f"Input (latent z): {x.shape}  [B, z_dim={z_dim}, T=5, H=8, W=8]")
print(f"{'='*70}")

# Step 1: conv1 -- z_dim to 4*dim channels
with torch.no_grad():
    x = dec.conv1(x)
assert x.shape == torch.Size([1, 128, 5, 8, 8])
print(f"Step 1 -- conv1 ({z_dim}->128):       {x.shape}")

# Step 2: Middle block
for layer in dec.middle:
    with torch.no_grad():
        x = layer(x)
assert x.shape == torch.Size([1, 128, 5, 8, 8])
print(f"Step 2 -- Middle block:           {x.shape}  (shape preserved)")

# Decode the upsamples layer by layer
# Level 0: 4 ResBlocks + 1 Resample = 5 layers
# Level 1: 4 ResBlocks + 1 Resample = 5 layers
# Level 2: 4 ResBlocks + 1 Resample = 5 layers
# Level 3: 4 ResBlocks = 4 layers
level_layers = list(dec.upsamples)
idx = 0

# Level 0: 4 ResBlocks(128->128) + Resample(128) -> 64 channels, 8->16 spatial
for layer in level_layers[idx:idx+5]:
    with torch.no_grad():
        x = layer(x)
idx += 5
assert x.shape == torch.Size([1, 64, 5, 16, 16])
print(f"Step 3 -- Level 0 + up3d:         {x.shape}  (128->64 via Resample halving, 8->16 spatial)")

# Level 1: ResBlock(64->128) + ResBlock(128->128) x3 + Resample(128) -> 64 channels
for layer in level_layers[idx:idx+5]:
    with torch.no_grad():
        x = layer(x)
idx += 5
assert x.shape == torch.Size([1, 64, 5, 32, 32])
print(f"Step 4 -- Level 1 + up3d:         {x.shape}  (64->128->64 via Resample, 16->32 spatial)")

# Level 2: ResBlock(64->64) x4 + Resample(64) -> 32 channels
for layer in level_layers[idx:idx+5]:
    with torch.no_grad():
        x = layer(x)
idx += 5
assert x.shape == torch.Size([1, 32, 5, 64, 64])
print(f"Step 5 -- Level 2 + up2d:         {x.shape}  (64->32 via Resample halving, 32->64 spatial)")

# Level 3: ResBlock(32->32) x4 (no upsample)
for layer in level_layers[idx:]:
    with torch.no_grad():
        x = layer(x)
assert x.shape == torch.Size([1, 32, 5, 64, 64])
print(f"Step 6 -- Level 3 (no up):        {x.shape}  (32->32, spatial preserved)")

# Step 7: Head (norm + SiLU + conv to 3 RGB)
with torch.no_grad():
    for layer in dec.head:
        x = layer(x)
assert x.shape == torch.Size([1, 3, 5, 64, 64])
print(f"Step 7 -- Head (32->3):           {x.shape}  (RGB output)")
print(f"{'='*70}")
print(f"\nSpatial upsampling: 8x8 -> 64x64 (8x per dimension)")
print(f"Channel reduction: 128 -> 3 (through Resample halving + head)")

### Shape Trace Summary

| Step | Operation | Output Shape | Spatial | Channels | Note |
|------|-----------|-------------|---------|----------|------|
| Input | -- | [1, 4, 5, 8, 8] | 8x8 | 4 (z_dim) | |
| 1 | conv1 | [1, 128, 5, 8, 8] | 8x8 | 128 (4*dim) | |
| 2 | Middle | [1, 128, 5, 8, 8] | 8x8 | 128 | |
| 3 | Level 0 + up3d | [1, 64, 5, 16, 16] | 16x16 | 64 | Resample halves: 128->64 |
| 4 | Level 1 + up3d | [1, 64, 5, 32, 32] | 32x32 | 64 | 64->128 via ResBlock, Resample halves: 128->64 |
| 5 | Level 2 + up2d | [1, 32, 5, 64, 64] | 64x64 | 32 | Resample halves: 64->32 |
| 6 | Level 3 | [1, 32, 5, 64, 64] | 64x64 | 32 | No upsample, channels stay 32 |
| 7 | Head | [1, 3, 5, 64, 64] | 64x64 | 3 (RGB) | |

In [ ]:
# Verify: full forward pass matches step-by-step trace
z_test = torch.randn(1, z_dim, 5, 8, 8)
with torch.no_grad():
    out_dec = dec(z_test)

assert out_dec.shape == torch.Size([1, 3, 5, 64, 64]), f"Expected [1,3,5,64,64], got {out_dec.shape}"
print(f"Full decoder forward: {z_test.shape} -> {out_dec.shape}")
print(f"Decompression ratio:")
print(f"  Spatial: 8x8 -> 64x64 = 64x expansion per frame")
print(f"  Channels: {z_dim} -> 3 (z_dim -> RGB)")
print(f"  Total voxels: {z_dim*5*8*8:,} -> {3*5*64*64:,} = {3*5*64*64 / (z_dim*5*8*8):.1f}x expansion")

## 3. Encoder vs Decoder Symmetry

| Property | Encoder3d | Decoder3d |
|----------|-----------|-----------|  
| Input | [B, 3, T, H, W] | [B, z_dim, T, H/8, W/8] |
| conv1 | 3 -> dim (32) | z_dim -> 4*dim (128) |
| Levels | 4 (downsample at 0,1,2) | 4 (upsample at 0,1,2) |
| ResBlocks/level | 2 | num_res_blocks+1 = 4 |
| Resample direction | downsample (stride 2) | upsample (nearest + conv) |
| Resample channel effect | Preserved | Halved |
| Middle | ResBlock + Attn + ResBlock | ResBlock + Attn + ResBlock |
| Head | 4*dim -> z_dim (output channels) | dim -> 3 (RGB) |
| Output | [B, z_dim, T, H/8, W/8] | [B, 3, T, H, W] |

**Key differences:**
- Decoder uses `num_res_blocks+1` ResBlocks per level (4 with config=3) vs encoder's `num_res_blocks` (2) -- more capacity for reconstruction
- Encoder preserves channels at Resample, decoder halves them -- different Conv2d configs inside Resample
- Note: Encoder is instantiated with `z_dim*2=8` (VideoVAE_ pattern) so it outputs 8 channels for mu/log_var split; the decoder takes `z_dim=4` directly

In [ ]:
# Source: wan_video_vae.py, line 971 (VideoVAE_ encoder instantiation)
# Compare parameter counts
# Encoder uses z_dim*2=8 per production pattern (see NB-09)
enc = Encoder3d(dim=32, z_dim=8, dim_mult=[1,2,4,4],
                num_res_blocks=2, temperal_downsample=[False, True, True])

enc_params = sum(p.numel() for p in enc.parameters())
dec_params = sum(p.numel() for p in dec.parameters())

print(f"Encoder parameters: {enc_params:,}  (z_dim=8 per VideoVAE_ pattern)")
print(f"Decoder parameters: {dec_params:,}  (z_dim=4)")
print(f"Decoder / Encoder ratio: {dec_params / enc_params:.2f}x")
print(f"\nDecoder is larger because:")
print(f"  - {num_res_blocks+1} ResBlocks/level vs encoder's 2 (range(num_res_blocks+1))")
print(f"  - Reconstruction needs more capacity than compression")

## 4. Full Round-Trip: Encode -> Reparameterize -> Decode

This section verifies end-to-end dimensional consistency:

**Pipeline:** video [B,3,T,H,W] -> Encoder3d -> [B,z_dim*2,T,H/8,W/8] -> chunk -> mu,log_var -> z=mu (deterministic) -> Decoder3d -> [B,3,T,H,W]

We use `z=mu` (no sampling) for deterministic shape verification.

The key assertion: **output shape == input shape** (dimensions preserved through encode/decode).

Note: Pixel values will NOT match (lossy compression + random init weights), but SHAPES must match.

> **Important:** The encoder is instantiated with `z_dim*2=8` following the production VideoVAE_
> pattern (line 971). This means the encoder outputs 8 channels, which are split into
> mu (4 channels) and log_var (4 channels). The decoder takes z_dim=4 as input.

In [ ]:
# Source: wan_video_vae.py (VideoVAE_ lines 951-1039)
# Full round-trip verification: encode -> reparameterize -> decode

print("--- Full Round-Trip Verification ---\n")

# Step 0: Original input
original_input = torch.randn(1, 3, 5, 64, 64)
print(f"1. Original input:     {original_input.shape}  [B, RGB, T, H, W]")

# Step 1: Encode
# Encoder uses z_dim*2=8 per production VideoVAE_ pattern (line 971)
enc_rt = Encoder3d(dim=32, z_dim=8, dim_mult=[1,2,4,4],
                   num_res_blocks=2, temperal_downsample=[False, True, True])
enc_rt.eval()
with torch.no_grad():
    encoder_out = enc_rt(original_input)
assert encoder_out.shape == torch.Size([1, 8, 5, 8, 8])
print(f"2. Encoder output:     {encoder_out.shape}  [B, z_dim*2=8, T, H/8, W/8]")

# Step 2: Reparameterize (deterministic: z = mu)
mu, log_var = encoder_out.chunk(2, dim=1)
z = mu  # deterministic for shape verification
assert mu.shape == torch.Size([1, 4, 5, 8, 8])
assert log_var.shape == torch.Size([1, 4, 5, 8, 8])
print(f"3. Latent z (mu):      {z.shape}  [B, z_dim=4, T, H/8, W/8]")

# Step 3: Decode
dec_rt = Decoder3d(dim=32, z_dim=4, dim_mult=[1,2,4,4],
                   num_res_blocks=3, temperal_upsample=[True, True, False])
dec_rt.eval()
with torch.no_grad():
    decoded = dec_rt(z)
print(f"4. Decoder output:     {decoded.shape}  [B, RGB, T, H, W]")

# Step 4: Verify shape match
assert decoded.shape == original_input.shape, (
    f"Round-trip FAILED: {decoded.shape} != {original_input.shape}")
print(f"\n{'='*55}")
print(f"Round-trip shape PRESERVED: {decoded.shape} == {original_input.shape}")
print(f"{'='*55}")

# Note: values differ (random weights, lossy compression)
print(f"\nNote: Pixel values differ (random weights, lossy compression)")
print(f"  Input mean: {original_input.mean():.4f}, Decoded mean: {decoded.mean():.4f}")
print(f"  This is expected -- we verify SHAPE consistency, not value reconstruction.")

## 5. The Complete VAE Data Flow

```
COMPLETE VAE DATA FLOW (connecting to DiT)
============================================

       ENCODE (NB-09)                    DECODE (this notebook)
       ============                      ===========

Video [B,3,T,64,64]                     Latent [B,4,T,8,8]
       |                                       |
       v                                       v
  Encoder3d                              Decoder3d
  (4-level spatial                       (4-level spatial
   downsample)                            upsample + channel halving)
       |                                       |
       v                                       v
  [B, 8, T, 8, 8]                       Video [B,3,T,64,64]
  (z_dim*2 channels)                     (reconstructed)
       |
       v
  chunk(2, dim=1)
       |
       +-> mu [B,4,T,8,8]
       +-> log_var [B,4,T,8,8]
       |
       v  reparameterize
  z [B,4,T,8,8]
       |
       v  normalize: (z - mean) / std
  z_norm [B,4,T,8,8]
       |
       v
  +-----------+
  |    DiT    |  <-- Processes latents (Phase 2: WanModel)
  | (denoise) |
  +-----------+
       |
       v  denormalize: z * std + mean
  z_denoised [B,4,T,8,8] -----------> Decoder3d input (above)
```

This diagram shows where the VAE sits in the overall pipeline: it provides the
representation space that the diffusion model operates in.

In [ ]:
# Summary: VAE compression statistics with reduced config
print("VAE Compression Summary (reduced config)")
print(f"{'='*50}")
print(f"\nInput:  [1, 3, 5, 64, 64]  = {3*5*64*64:>8,} values")
print(f"Latent: [1, 4, 5, 8, 8]   = {4*5*8*8:>8,} values")
print(f"Compression ratio: {3*5*64*64 / (4*5*8*8):.1f}x")
print(f"\nProduction (dim=96, z_dim=16, 480x832, T_latent=6):")
prod_input = 3 * 21 * 480 * 832
prod_latent = 16 * 6 * 60 * 104  # H/8, W/8, T_latent=6
print(f"Input:  [1, 3, 21, 480, 832]    = {prod_input:>12,} values")
print(f"Latent: [1, 16, 6, 60, 104]     = {prod_latent:>12,} values")
print(f"Compression ratio: {prod_input / prod_latent:.1f}x")
print(f"\nBreakdown of {prod_input / prod_latent:.0f}x compression:")
print(f"  Spatial: 8x (per dimension) -> 64x (H*W)")
print(f"  Temporal: ~3.5x (21 -> 6 frames via chunked inference)")
print(f"  Channels: 3 -> 16 (expansion, but 64*3.5/16 = 14x per channel)")

# Model sizes
enc_params = sum(p.numel() for p in enc_rt.parameters())
dec_params = sum(p.numel() for p in dec_rt.parameters())
print(f"\nModel sizes (reduced config):")
print(f"  Encoder: {enc_params:,} parameters")
print(f"  Decoder: {dec_params:,} parameters")
print(f"  Total VAE: {enc_params + dec_params:,} parameters")

## Summary

### Key Takeaways

- **Decoder3d** reverses the encoder's 4-level spatial downsampling with progressive upsampling: 8x8 -> 16x16 -> 32x32 -> 64x64
- **Resample channel halving** is the key decoder pattern: `Resample(upsample)` internally applies `Conv2d(dim, dim//2)`, so each level receives half the channels. The decoder compensates with `in_dim //= 2` at levels 1-3 (lines 770-771).
- **`num_res_blocks + 1` per level:** The inner loop at line 772 runs `range(num_res_blocks + 1)`, giving 4 ResBlocks per level when `num_res_blocks=3` (vs encoder's 2 per level)
- **Round-trip verification** confirms encode -> decode preserves dimensions: [1,3,5,64,64] -> latent -> [1,3,5,64,64]
- **Decoder is larger** than encoder (4 ResBlocks/level vs 2) -- reconstruction needs more capacity

### Source References

| Symbol | Location |
|--------|----------|
| Decoder3d | `wan_video_vae.py`, line 736 |
| Decoder3d.forward | `wan_video_vae.py`, line 789 |
| Channel compensation (in_dim //= 2) | `wan_video_vae.py`, lines 770-771 |
| Inner loop (num_res_blocks + 1) | `wan_video_vae.py`, line 772 |
| Resample (upsample2d/3d) | `wan_video_vae.py`, lines 82-174 |
| VideoVAE_ (full pipeline) | `wan_video_vae.py`, line 951 |

## Exercises

### Exercise 1 -- Channel Compensation Experiment
What happens if you remove the `in_dim = in_dim // 2` compensation (lines 770-771)?
Try instantiating Decoder3d with a modified dim_mult that accounts for this differently.
Hint: Without the compensation, the first ResBlock at each level would have a shape mismatch
between its input and the expected `in_dim`.

### Exercise 2 -- ResBlock Count Analysis
The decoder uses `num_res_blocks=3` while the encoder uses `num_res_blocks=2`.
But the decoder loop runs `range(num_res_blocks + 1)` while the encoder runs `range(num_res_blocks)`.
Calculate how many total ResidualBlocks are in each (count all levels + middle).
Why might reconstruction need more capacity than compression?

### Exercise 3 -- Alternative dim_mult
Trace the channel flow for a hypothetical decoder with `dim_mult=[1,2,4,8]`.
What would the `dims` list be? What would the channel counts be at each level after
Resample halving? Would any level need a shortcut projection in its first ResidualBlock
(i.e., where `in_dim != out_dim`)?